# Engenharia de Features (v2)
## Com ELO Rating e Separação Amistosos / Competitivos

**Melhorias em relação à v1:**
- **ELO Rating histórico** — pondera gols pelo nível do adversário. Um gol contra a Alemanha (ELO alto) vale mais que um gol contra San Marino (ELO baixo)
- **Separação de amistosos e competitivos** — features calculadas separadamente para cada tipo de jogo

**Referência:** Hvattum & Arntzen (2010) — *Using ELO ratings for match result prediction in association football*

**Dataset de entrada:** `data/raw/results.csv`  
**Dataset de saída:** `data/processed/features_completo_v2.csv`

## 1. Imports e Carregamento

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../src/features')
from elo import calcular_elo_historico

df_raw = pd.read_csv('../data/raw/results.csv', parse_dates=['date'])
print(f'Shape: {df_raw.shape}')
df_raw.head()

Shape: (49287, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


## 2. Calculando o ELO Histórico

O ELO é calculado para **todos os jogos desde 1872** — quanto mais jogos processados antes do período de interesse, mais preciso o ELO de cada seleção.

- **K=40** para jogos competitivos (Eliminatórias, Copa América, Eurocopa etc.)
- **K=20** para amistosos — menor impacto no ELO
- Seleções sem histórico começam com **ELO=1000**

In [2]:
df = calcular_elo_historico(df_raw, elo_inicial=1000, k_competitivo=40, k_amistoso=20)

# Salvar dataset com ELO para reutilização
df.to_csv('../data/processed/results_com_elo.csv', index=False)

print(f'ELO calculado para {len(df)} jogos')
print(f'\nExemplo — últimos jogos do Brasil:')
brasil = df[(df['home_team'] == 'Brazil') | (df['away_team'] == 'Brazil')].tail(5)
print(brasil[['date', 'home_team', 'away_team', 'home_score', 'away_score',
              'elo_home_antes', 'elo_away_antes']].to_string(index=False))

ELO calculado para 49287 jogos

Exemplo — últimos jogos do Brasil:
      date home_team away_team  home_score  away_score  elo_home_antes  elo_away_antes
2026-03-26    Brazil    France         1.0         2.0     1420.890606     1500.007230
2026-03-31    Brazil   Croatia         3.0         1.0     1413.129204     1405.719685
2026-06-13    Brazil   Morocco         NaN         NaN     1422.915974     1427.321877
2026-06-19    Brazil     Haiti         NaN         NaN     1403.169584     1117.239298
2026-06-24  Scotland    Brazil         NaN         NaN     1273.495100     1369.635883


## 3. Funções de Feature Engineering

### 3.1 Função de features por seleção

Para cada seleção e ciclo, calculamos:
- Features do **ciclo completo** (todos os jogos)
- Features dos **últimos 15 jogos**
- Features separadas por **tipo** (competitivo vs amistoso)
- Features **ponderadas pelo ELO** do adversário

In [3]:
def extrair_stats(jogos, selecao):
    """Extrai gols marcados, sofridos, vitórias e ELO adversário para uma seleção."""
    gm, gs, vit, elo_adv = [], [], [], []
    for _, row in jogos.iterrows():
        if row['home_team'] == selecao:
            gm.append(row['home_score'])
            gs.append(row['away_score'])
            vit.append(1 if row['home_score'] > row['away_score'] else 0)
            elo_adv.append(row['elo_away_antes'])
        else:
            gm.append(row['away_score'])
            gs.append(row['home_score'])
            vit.append(1 if row['away_score'] > row['home_score'] else 0)
            elo_adv.append(row['elo_home_antes'])
    return np.array(gm), np.array(gs), np.array(vit), np.array(elo_adv)


def calcular_features_v2(selecao, ciclo, copa):
    """Calcula features v2 com ELO e separação amistoso/competitivo."""

    # Todos os jogos da seleção no ciclo
    jogos = ciclo[
        (ciclo['home_team'] == selecao) |
        (ciclo['away_team'] == selecao)
    ].sort_values('date')

    # Separação por tipo
    jogos_comp = jogos[jogos['tournament'] != 'Friendly']
    jogos_ami  = jogos[jogos['tournament'] == 'Friendly']

    # Stats do ciclo completo
    gm, gs, vit, elo_adv = extrair_stats(jogos, selecao)

    # Stats dos últimos 15 jogos
    ultimos15 = jogos.tail(15)
    gm15, gs15, vit15, elo_adv15 = extrair_stats(ultimos15, selecao)

    # Stats competitivos
    if len(jogos_comp) > 0:
        gm_c, gs_c, vit_c, elo_c = extrair_stats(jogos_comp, selecao)
        media_gm_comp = gm_c.mean()
        media_gs_comp = gs_c.mean()
        pct_vit_comp  = vit_c.mean()
    else:
        media_gm_comp = gm.mean()
        media_gs_comp = gs.mean()
        pct_vit_comp  = vit.mean()

    # Stats amistosos
    if len(jogos_ami) > 0:
        gm_a, gs_a, vit_a, elo_a = extrair_stats(jogos_ami, selecao)
        media_gm_ami = gm_a.mean()
    else:
        media_gm_ami = gm.mean()

    # Gols ponderados pelo ELO do adversário (normalizado por 1000)
    gols_ponderados      = (gm * (elo_adv / 1000)).mean()
    gols_ponderados_ult15 = (gm15 * (elo_adv15 / 1000)).mean()

    # Target
    selecao_copa = copa[
        (copa['home_team'] == selecao) |
        (copa['away_team'] == selecao)
    ]
    gols_copa = [
        row['home_score'] if row['home_team'] == selecao else row['away_score']
        for _, row in selecao_copa.iterrows()
    ]

    return {
        # Features v1 (mantidas)
        'media_gols_marcados_ciclo':  gm.mean(),
        'media_gols_sofridos_ciclo':  gs.mean(),
        'pct_vitorias_ciclo':         vit.mean(),
        'total_jogos_ciclo':          len(jogos),
        'media_gols_marcados_ult15':  gm15.mean(),
        'media_gols_sofridos_ult15':  gs15.mean(),
        'pct_vitorias_ult15':         vit15.mean(),
        # Features v2 — novas
        'media_gols_competitivos':    media_gm_comp,
        'media_gols_sofridos_comp':   media_gs_comp,
        'pct_vitorias_comp':          pct_vit_comp,
        'media_gols_amistosos':       media_gm_ami,
        'gols_ponderados_elo_ciclo':  gols_ponderados,
        'gols_ponderados_elo_ult15':  gols_ponderados_ult15,
        # Target
        'media_gols_copa':            np.mean(gols_copa)
    }

print('Funções definidas com sucesso!')

Funções definidas com sucesso!


### 3.2 Teste com o Brasil — Copa 2022

In [4]:
copa_2022 = df[
    (df['tournament'] == 'FIFA World Cup') &
    (df['date'].dt.year == 2022)
]

ciclo_2022 = df[
    (df['date'] >= '2018-07-16') &
    (df['date'] <= '2022-11-19') &
    (df['tournament'] != 'FIFA World Cup')
]

resultado_brasil = calcular_features_v2('Brazil', ciclo_2022, copa_2022)
print('Features v2 — Brasil (Copa 2022):')
for k, v in resultado_brasil.items():
    print(f'  {k:<35} {v:.4f}')

Features v2 — Brasil (Copa 2022):
  media_gols_marcados_ciclo           2.2200
  media_gols_sofridos_ciclo           0.3800
  pct_vitorias_ciclo                  0.7400
  total_jogos_ciclo                   50.0000
  media_gols_marcados_ult15           2.5333
  media_gols_sofridos_ult15           0.3333
  pct_vitorias_ult15                  0.8000
  media_gols_competitivos             2.0625
  media_gols_sofridos_comp            0.3125
  pct_vitorias_comp                   0.7500
  media_gols_amistosos                2.5000
  gols_ponderados_elo_ciclo           2.7696
  gols_ponderados_elo_ult15           3.1818
  media_gols_copa                     1.6000


## 4. Pipeline Completo — Todas as Copas (1994–2022)

In [5]:
copas = {
    1994: {'ciclo_inicio': '1990-07-09', 'ciclo_fim': '1994-06-16', 'copa_inicio': '1994-06-17', 'copa_fim': '1994-07-17'},
    1998: {'ciclo_inicio': '1994-07-18', 'ciclo_fim': '1998-06-09', 'copa_inicio': '1998-06-10', 'copa_fim': '1998-07-12'},
    2002: {'ciclo_inicio': '1998-07-13', 'ciclo_fim': '2002-05-30', 'copa_inicio': '2002-05-31', 'copa_fim': '2002-06-30'},
    2006: {'ciclo_inicio': '2002-07-01', 'ciclo_fim': '2006-06-08', 'copa_inicio': '2006-06-09', 'copa_fim': '2006-07-09'},
    2010: {'ciclo_inicio': '2006-07-10', 'ciclo_fim': '2010-06-10', 'copa_inicio': '2010-06-11', 'copa_fim': '2010-07-11'},
    2014: {'ciclo_inicio': '2010-07-12', 'ciclo_fim': '2014-06-11', 'copa_inicio': '2014-06-12', 'copa_fim': '2014-07-13'},
    2018: {'ciclo_inicio': '2014-07-14', 'ciclo_fim': '2018-06-13', 'copa_inicio': '2018-06-14', 'copa_fim': '2018-07-15'},
    2022: {'ciclo_inicio': '2018-07-16', 'ciclo_fim': '2022-11-19', 'copa_inicio': '2022-11-20', 'copa_fim': '2022-12-18'},
}

todos_dados = []

for ano, datas in copas.items():
    print(f'Processando Copa {ano}...')

    ciclo = df[
        (df['date'] >= datas['ciclo_inicio']) &
        (df['date'] <= datas['ciclo_fim']) &
        (df['tournament'] != 'FIFA World Cup')
    ]

    copa = df[
        (df['tournament'] == 'FIFA World Cup') &
        (df['date'] >= datas['copa_inicio']) &
        (df['date'] <= datas['copa_fim'])
    ]

    selecoes = pd.unique(copa[['home_team', 'away_team']].values.ravel())

    for selecao in selecoes:
        try:
            resultado = calcular_features_v2(selecao, ciclo, copa)
            resultado['selecao']   = selecao
            resultado['copa_alvo'] = ano
            todos_dados.append(resultado)
        except Exception as e:
            print(f'  Erro em {selecao}: {e}')

df_final = pd.DataFrame(todos_dados)
print(f'\nDataset final: {df_final.shape[0]} linhas × {df_final.shape[1]} colunas')
df_final.head()

Processando Copa 1994...
Processando Copa 1998...
Processando Copa 2002...
Processando Copa 2006...
Processando Copa 2010...
Processando Copa 2014...
Processando Copa 2018...
Processando Copa 2022...

Dataset final: 248 linhas × 16 colunas


,media_gols_marcados_ciclo,media_gols_sofridos_ciclo,pct_vitorias_ciclo,total_jogos_ciclo,media_gols_marcados_ult15,media_gols_sofridos_ult15,pct_vitorias_ult15,media_gols_competitivos,media_gols_sofridos_comp,pct_vitorias_comp,media_gols_amistosos,gols_ponderados_elo_ciclo,gols_ponderados_elo_ult15,media_gols_copa,selecao,copa_alvo
0,1.975610,0.975610,0.609756,41,2.533333,1.066667,0.666667,2.071429,1.357143,0.642857,1.925926,2.252097,2.976510,1.800000,Germany,1994
1,1.023810,1.190476,0.214286,42,0.600000,1.133333,0.133333,1.304348,1.304348,0.260870,0.684211,1.074301,0.688356,0.333333,Bolivia,1994
2,1.885714,0.914286,0.514286,35,2.266667,0.733333,0.666667,2.315789,0.842105,0.578947,1.375000,1.969742,2.317873,2.000000,Spain,1994
3,1.750000,0.625000,0.517857,56,1.533333,1.066667,0.333333,2.055556,0.444444,0.583333,1.200000,1.791800,1.788201,1.333333,South Korea,1994
4,1.279070,0.604651,0.465116,43,1.733333,0.400000,0.600000,1.333333,0.625000,0.458333,1.210526,1.452326,1.920815,1.333333,Colombia,1994


## 5. Validação e Salvamento

In [6]:
print('Linhas por Copa:')
print(df_final['copa_alvo'].value_counts().sort_index())

print('\nEstatísticas do target:')
print(df_final['media_gols_copa'].describe().round(3))

print('\nNovas features — exemplo Brasil 2022:')
brasil_2022 = df_final[
    (df_final['selecao'] == 'Brazil') &
    (df_final['copa_alvo'] == 2022)
][['media_gols_marcados_ciclo', 'media_gols_competitivos',
   'media_gols_amistosos', 'gols_ponderados_elo_ciclo',
   'gols_ponderados_elo_ult15']]
print(brasil_2022.to_string(index=False))

Linhas por Copa:
copa_alvo
1994    24
1998    32
2002    32
2006    32
2010    32
2014    32
2018    32
2022    32
Name: count, dtype: int64

Estatísticas do target:
count    248.000
mean       1.166
std        0.591
min        0.000
25%        0.667
50%        1.143
75%        1.571
max        2.667
Name: media_gols_copa, dtype: float64

Novas features — exemplo Brasil 2022:
 media_gols_marcados_ciclo  media_gols_competitivos  media_gols_amistosos  gols_ponderados_elo_ciclo  gols_ponderados_elo_ult15
                      2.22                   2.0625                   2.5                   2.769584                   3.181826


In [7]:
df_final.to_csv('../data/processed/features_completo_v2.csv', index=False)
print('Dataset v2 salvo em: data/processed/features_completo_v2.csv')

Dataset v2 salvo em: data/processed/features_completo_v2.csv


## 6. Resumo das Features

| Feature | Versão | Descrição |
|---------|--------|-----------|
| `media_gols_marcados_ciclo` | v1 | Média de gols marcados no ciclo completo |
| `media_gols_sofridos_ciclo` | v1 | Média de gols sofridos no ciclo completo |
| `pct_vitorias_ciclo` | v1 | % de vitórias no ciclo |
| `total_jogos_ciclo` | v1 | Total de jogos no ciclo |
| `media_gols_marcados_ult15` | v1 | Média de gols nos últimos 15 jogos |
| `media_gols_sofridos_ult15` | v1 | Média de gols sofridos nos últimos 15 |
| `pct_vitorias_ult15` | v1 | % de vitórias nos últimos 15 jogos |
| `media_gols_competitivos` | **v2** | Média de gols apenas em jogos competitivos |
| `media_gols_sofridos_comp` | **v2** | Média de gols sofridos em competitivos |
| `pct_vitorias_comp` | **v2** | % de vitórias em jogos competitivos |
| `media_gols_amistosos` | **v2** | Média de gols apenas em amistosos |
| `gols_ponderados_elo_ciclo` | **v2** | Gols × (ELO adversário / 1000) — ciclo |
| `gols_ponderados_elo_ult15` | **v2** | Gols × (ELO adversário / 1000) — últ. 15 |
| `media_gols_copa` | target | Média de gols marcados na Copa |

**Próximo passo:** `03_modelos.ipynb` — comparar v1 vs v2 e verificar se as novas features melhoram o MAE.